Created Train/Test split with the latest month (September 2026) being the test split and the year before being the train split.

In [46]:
from pathlib import Path
import os
import pandas as pd

DATA_DIR = Path(os.getenv("IDX_DATA_DIR", "data"))
CSV_DIR = DATA_DIR / "raw/csv"

from pathlib import Path
import os
import pandas as pd

DATA_DIR = Path(os.getenv("IDX_DATA_DIR", "data"))
CSV_DIR = DATA_DIR / "raw/csv"

TEST_MONTH = "202609"
TRAIN_MONTHS = [
    "202509", "202510", "202511", "202512",
    "202601", "202602", "202603", "202604",
    "202605", "202606", "202607", "202608",
]

In [47]:
def sold_path(month):
    generated = CSV_DIR / f"GeneratedCRMLSSold{month}.csv"
    raw = CSV_DIR / f"CRMLSSold{month}.csv"
    if generated.exists():
        return generated
    return raw


def load_month(month):
    df = pd.read_csv(sold_path(month), low_memory=False)
    df["month"] = month
    return df


train = pd.concat([load_month(m) for m in TRAIN_MONTHS], ignore_index=True)
test = load_month(TEST_MONTH)

print("train months", TRAIN_MONTHS)
print(train["month"].value_counts().sort_index())
print(test["month"].value_counts().sort_index())

train months ['202509', '202510', '202511', '202512', '202601', '202602', '202603', '202604', '202605', '202606', '202607', '202608']
month
202509    22443
202510    23233
202511    19088
202512    20538
202601    16487
202602    19010
202603    23372
202604    24261
202605    24277
202606    25812
202607    25467
202608    23133
Name: count, dtype: int64
month
202609    22107
Name: count, dtype: int64


Combining test/train tables into one in order to handle null values and encode categorical columns. 

In [48]:
df = pd.concat(
    [train.assign(split="train"), test.assign(split="test")],
    ignore_index=True,
)
df["split"].value_counts()

split
train    267121
test      22107
Name: count, dtype: int64

Keep the listing features used for modeling. Everything else (sale-only fields, agent/buyer text, mostly empty columns) is left out.

In [49]:
keep_cols = [
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "GarageSpaces",
    "PropertyType",
    "PropertySubType",
    "WaterfrontYN",
    "BasementYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "PoolPrivateYN",
    "ViewYN",
    "NewConstructionYN",
    "City",
    "PostalCode",
    "Latitude",
    "Longitude",
    "month",
    "split",
]
df = df[keep_cols]

Determining the proportion of entries that have a null value for each column

In [50]:
null_pct = df.isnull().mean() * 100
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
null_pct

MiddleOrJuniorSchoolDistrict    100.000000
ElementarySchoolDistrict        100.000000
CoveredSpaces                   100.000000
AboveGradeFinishedArea          100.000000
FireplacesTotal                 100.000000
                                   ...    
PostalCode                        0.027660
City                              0.010372
ListAgentLastName                 0.010027
ClosePrice                        0.001729
StateOrProvince                   0.000691
Length: 66, dtype: float64

Drop columns that are null for every row, observe remaining columns

In [51]:
all_null = df.columns[df.isna().all()]
df = df.drop(columns=all_null)

null_pct = df.isnull().mean() * 100
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
print(null_pct.to_string())

WaterfrontYN                99.943989
TaxYear                     99.939148
BusinessType                99.742072
TaxAnnualAmount             99.645954
BelowGradeFinishedArea      99.480687
BasementYN                  98.363229
BuilderName                 96.286667
LotSizeDimensions           94.530958
CoBuyerAgentFirstName       91.491488
ElementarySchool            89.583650
MiddleOrJuniorSchool        89.481655
BuildingAreaTotal           86.603303
HighSchool                  86.197740
CoListAgentFirstName        78.813946
CoListAgentLastName         78.767270
CoListOfficeName            77.215553
AssociationFeeFrequency     70.411578
SubdivisionName             63.902181
MainLevelBedrooms           45.783949
ListAgentEmail              41.921598
Flooring                    41.415423
OriginatingSystemSubName    35.193342
OriginatingSystemName       35.193342
HighSchoolDistrict          34.165433
AssociationFee              30.470079
AttachedGarageYN            28.433969
Stories     

See the values a column takes, including null

For waterfront column, notice that the only non-null value is true meaning we can impute the null values to be false and then encode the columns by making true = 1 and false = 0

In [52]:
def colValues(col):
    counts = df[col].value_counts(dropna=False)
    print(f"{len(counts)} distinct values")
    print(counts.to_string())

colValues("WaterfrontYN")

2 distinct values
WaterfrontYN
NaN     289066
True       162


In [53]:
df["WaterfrontYN"] = df["WaterfrontYN"].eq(True).astype(int)

In [ ]:
colValues("BasementYN")
colValues("AttachedGarageYN")
colValues("WaterfrontYN")
colValues("WaterfrontYN")